# Kvasir-SEG SegFormer Baseline

SegFormer-based segmentation baseline.\nAttempts pretrained load first; falls back to a local random-init config if offline.

In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)

BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:

import os
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import SegformerConfig, SegformerForSemanticSegmentation

from utils.segmentation_common import (
    find_kvasir_seg_root,
    load_metadata,
    TrainConfig,
    train_and_evaluate,
)

ROOT = find_kvasir_seg_root()
META_CSV = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'metadata_enriched.csv'
SPLIT_HASH_TXT = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'split_hash.txt'
OUT_DIR = ROOT / '2_modern_segmentation' / 'out' / 'segformer_b2_finetune'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.getenv('SEED', '42'))
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '2'))
NUM_WORKERS = int(os.getenv('NUM_WORKERS', '2'))
EPOCHS = int(os.getenv('EPOCHS', '4'))
LR = float(os.getenv('LR', '1e-4'))
WEIGHT_DECAY = float(os.getenv('WEIGHT_DECAY', '1e-4'))
IMAGE_SIZE = int(os.getenv('IMAGE_SIZE', '224'))
THRESHOLD = float(os.getenv('THRESHOLD', '0.5'))

MAX_TRAIN = int(os.getenv('MAX_TRAIN_SAMPLES', '0')) or None
MAX_VAL = int(os.getenv('MAX_VAL_SAMPLES', '0')) or None
MAX_TEST = int(os.getenv('MAX_TEST_SAMPLES', '0')) or None

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)
print('CUDA available:', torch.cuda.is_available())


2026-02-15 10:37:19.783085: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
OUT_DIR: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/2_modern_segmentation/out/segformer_b2_finetune
CUDA available: True


In [3]:
class SegformerBinaryWrapper(nn.Module):
    def __init__(self):
        super().__init__()
        self.source = 'random_init_config'
        model = None

        allow_hf_download = os.getenv('ALLOW_HF_DOWNLOAD', '0') == '1'

        # Prefer local/offline cached weights first
        try:
            model = SegformerForSemanticSegmentation.from_pretrained(
                'nvidia/segformer-b2-finetuned-ade-512-512',
                num_labels=1,
                ignore_mismatched_sizes=True,
                local_files_only=True,
            )
            self.source = 'cached_pretrained_b2'
        except Exception:
            pass

        # Optional online pull if explicitly enabled
        if model is None and allow_hf_download:
            try:
                model = SegformerForSemanticSegmentation.from_pretrained(
                    'nvidia/segformer-b2-finetuned-ade-512-512',
                    num_labels=1,
                    ignore_mismatched_sizes=True,
                )
                self.source = 'downloaded_pretrained_b2'
            except Exception:
                pass

        # Safe fallback: small random-init config
        if model is None:
            cfg = SegformerConfig(
                num_labels=1,
                depths=[2, 2, 2, 2],
                hidden_sizes=[32, 64, 160, 256],
                decoder_hidden_size=128,
            )
            model = SegformerForSemanticSegmentation(cfg)
            self.source = 'random_init_b0_like'

        self.model = model

    def forward(self, x):
        out = self.model(pixel_values=x)
        logits = out.logits
        logits = F.interpolate(logits, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return logits


meta_df = load_metadata(META_CSV)
split_hash = SPLIT_HASH_TXT.read_text().strip() if SPLIT_HASH_TXT.exists() else None

cfg = TrainConfig(
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    threshold=THRESHOLD,
    compute_hd95=False,
    save_pred_masks=True,
    pred_mask_limit=200,
)

model = SegformerBinaryWrapper()
print('SegFormer source:', model.source)
print('ALLOW_HF_DOWNLOAD:', os.getenv('ALLOW_HF_DOWNLOAD', '0'))

SegFormer source: random_init_b0_like
ALLOW_HF_DOWNLOAD: 0


In [4]:

results = train_and_evaluate(
    model=model,
    model_name='segformer_binary',
    root=ROOT,
    out_dir=OUT_DIR,
    cfg=cfg,
    metadata_df=meta_df,
    split_hash=split_hash,
    max_train=MAX_TRAIN,
    max_val=MAX_VAL,
    max_test=MAX_TEST,
)

print(json.dumps(results, indent=2))


Epoch 1/4 train_loss=0.6051 val_loss=0.5351 val_dice=0.4344
Epoch 2/4 train_loss=0.5232 val_loss=0.5187 val_dice=0.4037
Epoch 3/4 train_loss=0.4995 val_loss=0.5040 val_dice=0.4545
Epoch 4/4 train_loss=0.4854 val_loss=0.4670 val_dice=0.4476


/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/utils/segmentation_common.py:1390: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_st

{
  "train": {
    "n": 800,
    "dice_mean": 0.46128940735407525,
    "dice_median": 0.4657133105872579,
    "dice_std": 0.23677968699543128,
    "iou_mean": 0.3318808182531338,
    "iou_median": 0.3035373605454599,
    "iou_std": 0.21126364016815366,
    "precision_mean": 0.4089691339406984,
    "precision_median": 0.35085244866319976,
    "precision_std": 0.28658373951461436,
    "recall_mean": 0.7296664281089658,
    "recall_median": 0.8049653301287598,
    "recall_std": 0.26228587770232453,
    "f1_mean": 0.46128940351120595,
    "f1_median": 0.46571330579029835,
    "f1_std": 0.2367796860621272,
    "specificity_mean": 0.8333128148531697,
    "specificity_median": 0.8215266017041024,
    "specificity_std": 0.0820680175219675,
    "loss": 0.502201856225729
  },
  "val": {
    "n": 100,
    "dice_mean": 0.4544524653240382,
    "dice_median": 0.49570334746073663,
    "dice_std": 0.2695025150954488,
    "iou_mean": 0.3346906478446961,
    "iou_median": 0.32953132163552024,
    "iou_s